<a href="https://colab.research.google.com/github/urcraft/demi_api_lecture_notebooks/blob/main/flask_api_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyngrok

In [ ]:
# Step 1: Import necessary libraries
from flask import Flask, request, jsonify  # Flask for web framework, request to access incoming data, jsonify to convert Python to JSON
from pyngrok import ngrok  # pyngrok allows us to create a public URL for our local app
import os  # Operating system utilities, not essential for this app but often useful

# Step 2: Initialize our Flask application
app = Flask(__name__)  # Create a new Flask application

# Step 3: Create our "database" (just a Python list for this example)
todos = [
    {"id": 1, "title": "Learn REST APIs", "completed": False},
    {"id": 2, "title": "Build a Flask API", "completed": False},
    {"id": 3, "title": "Share with classmates", "completed": False}
]

# Step 4: Helper function to find a specific todo by ID
def find_todo(todo_id):
    # Loop through each todo in our list
    for todo in todos:
        # If we find the todo with the matching ID, return it
        if todo["id"] == todo_id:
            return todo
    # If we didn't find a matching todo, return None
    return None

# Step 5: GET endpoint to retrieve all todos (Read operation)
@app.route('/api/todos', methods=['GET'])  # Define the URL path and HTTP method
def get_todos():
    # Check if the user wants to filter by completion status
    completed_filter = request.args.get('completed')  # Get the 'completed' query parameter if it exists

    if completed_filter is not None:  # If the parameter was provided
        # Convert string 'true'/'false' to boolean
        is_completed = completed_filter.lower() == 'true'
        # Return only todos matching the completion status
        filtered_todos = [todo for todo in todos if todo['completed'] == is_completed]
        return jsonify(filtered_todos)  # Convert to JSON and return

    # If no filter was specified, return all todos
    return jsonify(todos)

# Step 6: GET endpoint to retrieve a specific todo by ID
@app.route('/api/todos/<int:todo_id>', methods=['GET'])
def get_todo(todo_id):
    todo = find_todo(todo_id)  # Use our helper function to find the todo
    if todo:  # If we found it
        return jsonify(todo)  # Return it as JSON
    # If we didn't find it, return an error message and 404 status code
    return jsonify({"error": "Todo not found"}), 404

# Step 7: POST endpoint to create a new todo (Create operation)
@app.route('/api/todos', methods=['POST'])
def create_todo():
    # Check if we received valid JSON with a title field
    if not request.json or 'title' not in request.json:
        return jsonify({"error": "Title is required"}), 400  # Bad request

    # Generate a new ID (one higher than the highest existing ID)
    next_id = max([todo["id"] for todo in todos], default=0) + 1

    # Create the new todo object
    new_todo = {
        "id": next_id,
        "title": request.json['title'],
        "completed": request.json.get('completed', False)  # Default to False if not provided
    }

    # Add it to our list of todos
    todos.append(new_todo)
    # Return the new todo with a 201 Created status code
    return jsonify(new_todo), 201

# Step 8: PUT endpoint to fully update a todo (Update operation)
@app.route('/api/todos/<int:todo_id>', methods=['PUT'])
def update_todo(todo_id):
    todo = find_todo(todo_id)  # Find the todo to update
    if not todo:  # If it doesn't exist
        return jsonify({"error": "Todo not found"}), 404

    # Validate the request data
    if not request.json:
        return jsonify({"error": "No data provided"}), 400

    if 'title' not in request.json:
        return jsonify({"error": "Title is required"}), 400

    # Update the todo with new values
    todo['title'] = request.json['title']
    todo['completed'] = request.json.get('completed', todo['completed'])

    # Return the updated todo
    return jsonify(todo)

# Step 9: PATCH endpoint to partially update a todo
@app.route('/api/todos/<int:todo_id>', methods=['PATCH'])
def patch_todo(todo_id):
    todo = find_todo(todo_id)  # Find the todo to update
    if not todo:  # If it doesn't exist
        return jsonify({"error": "Todo not found"}), 404

    if not request.json:  # If no data was provided
        return jsonify({"error": "No data provided"}), 400

    # Update only the fields that were provided in the request
    if 'title' in request.json:
        todo['title'] = request.json['title']

    if 'completed' in request.json:
        todo['completed'] = request.json['completed']

    # Return the updated todo
    return jsonify(todo)

# Step 10: DELETE endpoint to remove a todo (Delete operation)
@app.route('/api/todos/<int:todo_id>', methods=['DELETE'])
def delete_todo(todo_id):
    todo = find_todo(todo_id)  # Find the todo to delete
    if not todo:  # If it doesn't exist
        return jsonify({"error": "Todo not found"}), 404

    # Remove the todo from our list
    todos.remove(todo)
    # Return a success message
    return jsonify({"message": f"Todo {todo_id} deleted successfully"}), 200

# Step 11: Add a welcome route to explain the API
@app.route('/', methods=['GET'])
def welcome():
    return jsonify({
        "message": "Welcome to the Todo API!",
        "endpoints": {
            "GET /api/todos": "Get all todos",
            "GET /api/todos/:id": "Get a specific todo",
            "POST /api/todos": "Create a new todo",
            "PUT /api/todos/:id": "Update a todo (full replacement)",
            "PATCH /api/todos/:id": "Update a todo (partial update)",
            "DELETE /api/todos/:id": "Delete a todo"
        }
    })

In [ ]:
# Step 12: Set up ngrok to make our local app publicly accessible
def start_ngrok():
    # Collect the ngrok auth token from the user
    import getpass
    auth_token = getpass.getpass("Enter your ngrok auth token: ")
    ngrok.set_auth_token(auth_token)

    # Create a tunnel to expose port 5000 (where Flask runs)
    public_url = ngrok.connect(5000).public_url
    print(f" * ngrok tunnel available at: {public_url}")
    print(f" * Access the API at: {public_url}/api/todos")

    # Store the public URL in our app configuration
    app.config['BASE_URL'] = public_url

    return public_url

# Step 13: Main entry point to start our application
if __name__ == '__main__':
    # Start ngrok when app is run
    public_url = start_ngrok()

    # Run the Flask app on port 5000
    app.run(port=5000)